# Combined-property check: our equations vs the paper's models

The paper (Kang et al., *Comput. Mater. Sci.* 246 (2025) 113482) combines three properties. It looks for glasses with **high specific modulus E/ρ**, low density and low liquidus temperature, using predictions for the CaO–MgO–Al₂O₃–SiO₂ (CMSA) space. It then validates one glass experimentally.

This notebook compares our symbolic-regression equations with the paper's approach in three ways:

| Part | What is compared | Reference |
|---|---|---|
| A | The experimental glass 59SiO₂–16.5Al₂O₃–10CaO–14.5MgO | measured E = 93.6 GPa, ρ = 2.641 g/cm³ (paper) |
| B | **7,362 database glasses with both measured E and ρ**, giving a measured specific modulus | out-of-fold predictions from our equations and from the paper's models (random forest with the paper's hyperparameters), on the same folds |
| C | The 140,871-composition CMSA design space | the paper's own predictions (its `CMSA.csv`) |

**Requirements:**
- Run it in the `ml` folder, after `glass_property_pipeline_v4` has finished (it reads `results_v2/`).
- *Restart Kernel → Run All.*
- The paper-model cross-validation in Part B is saved fold by fold, so if it stops, Run All continues where it left off.

In [ ]:
from pathlib import Path
RESULTS_V2   = Path("results_v2")            # outputs of glass_property_pipeline_v4
OUT_DIR      = Path("results_combined")
DATA_DIR     = Path("data_cache")            # same cache the main notebook uses
OXIDE_TABLE  = Path("pure_oxide_properties.csv")
GITHUB_RAW   = "https://raw.githubusercontent.com/kzzzynobad/High-Specific-Modulus-Glass/main/DATASET"
SEED, OUTER_FOLDS = 42, 5                    # must match the main run (checked below)
N_JOBS = 4                                   # CPU threads for the paper's random forests (lower = less memory)
RUN_PAPER_MODELS = True                      # re-train the paper's models out-of-fold (Part B); ~10-30 min
TARGET_REGION = dict(max_density=2.65, min_E=100.0)   # the paper's Fig. 10 target window

DATASETS = {"youngs_modulus": ("YOUNGSMODULUS", "YoungModulus", "youngs_modulus_GPa", "e9843ed350120cd31fc482d6f2335f65"),
            "density":        ("DENSITY", "Density293K", "density_g_cm3", "b7da382a7058ea7b80441449bb0d6326"),
            "liquidus_temperature": ("TLIQUIDUS", "Tliquidus", "melting_point_K", "adece4e478c7181525861e514b37dedd")}
CMSA_MD5 = "6dd79d8f430ca7d30ed1db4d67fa7428"
OXIDES = ["Li2O","B2O3","Na2O","MgO","Al2O3","SiO2","P2O5","K2O","CaO","TiO2","Fe2O3","ZnO","SrO","Y2O3","ZrO2","SnO2","BaO"]
FEATURES = ["mean_p", "gmean_p", "hmean_p", "var_p", "S_mix"]
CONFIG_LOADED = True

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
import json, os, time, hashlib, gc, urllib.request, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sympy as sp
from scipy.optimize import curve_fit
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error
warnings.filterwarnings("ignore")
OUT_DIR.mkdir(exist_ok=True); (OUT_DIR / "checkpoints").mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10})

def md5(p): return hashlib.md5(Path(p).read_bytes()).hexdigest()
def fetch(rel, expected_md5):
    path = DATA_DIR / rel
    if not path.exists() or md5(path) != expected_md5:
        path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW}/{rel}".replace(" ", "%20")
        print("downloading", url); urllib.request.urlretrieve(url, path)
    assert md5(path) == expected_md5, f"{path}: checksum mismatch"
    return path

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"R2": r2_score(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))), "MAE": mean_absolute_error(y, p),
            "MedAE": median_absolute_error(y, p), "MAPE_%": float(np.mean(np.abs((y - p) / y)) * 100),
            "Spearman": float(spearmanr(y, p).correlation)}

def save_json(path, obj):
    tmp = Path(str(path) + ".tmp"); tmp.write_text(json.dumps(obj)); os.replace(tmp, path)

# --- data
raw = {}
for name, (folder, target, prop, h) in DATASETS.items():
    df = pd.read_csv(fetch(f"{folder}/CLEAN.csv", h)).rename(columns={"ZrO": "ZrO2"})
    raw[name] = df
cmsa = pd.read_csv(fetch("GENERATING DATASETS/CMSA.csv", CMSA_MD5)).rename(columns={"ZrO": "ZrO2"})
ox = pd.read_csv(OXIDE_TABLE).set_index("oxide")
assert ox[[v[2] for v in DATASETS.values()]].notna().all().all(), "oxide table still has gaps; use the filled table"
rr = json.loads((RESULTS_V2 / "run_record.json").read_text())
assert rr["config"]["SEED"] == SEED and rr["config"]["OUTER_FOLDS"] == OUTER_FOLDS and not rr["config"]["SELF_TEST"], "results_v2 is not the full run"
print({k: len(v) for k, v in raw.items()}, "| CMSA rows:", len(cmsa))

## Our models: descriptors, equations and out-of-fold predictions from the main run

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"

def descriptors(comp_df, p):
    X = comp_df[OXIDES].to_numpy(float); X = X / X.sum(axis=1, keepdims=True)
    pv = p.reindex(OXIDES).to_numpy(float); mean = X @ pv
    return pd.DataFrame({"mean_p": mean, "gmean_p": np.exp(X @ np.log(pv)), "hmean_p": 1 / (X @ (1 / pv)),
                         "var_p": np.clip(X @ pv**2 - mean**2, 0, None),
                         "S_mix": -(np.where(X > 0, X * np.log(np.where(X > 0, X, 1.0)), 0.0)).sum(axis=1)})

def grouped_kfold(groups, k, seed):        # identical to the main notebook
    uniq = np.unique(groups)
    fold_of_group = pd.Series(np.arange(len(uniq)) % k, index=np.random.default_rng(seed).permutation(uniq))
    fold = fold_of_group.reindex(groups).to_numpy()
    return [(np.where(fold != i)[0], np.where(fold == i)[0]) for i in range(k)]

D, folds, oof = {}, {}, {}
for name, (folder, target, prop, h) in DATASETS.items():
    D[name] = descriptors(raw[name], ox[prop])
    groups = pd.factorize(D[name].round(8).astype(str).agg("|".join, axis=1))[0]
    folds[name] = grouped_kfold(groups, OUTER_FOLDS, SEED)
    o = pd.read_csv(RESULTS_V2 / f"oof_predictions_{name}.csv")
    assert np.allclose(o["y_true"], raw[name][target]), f"{name}: saved predictions are not aligned with the data"
    oof[name] = {"sr_long (ours)": o["symbolic_regression"].to_numpy()}

# long final equations (all data) -> functions of the descriptors
final_eq = pd.read_csv(RESULTS_V2 / "final_equations.csv").set_index("property")
def eq_fn(text):
    f = sp.lambdify([sp.Symbol(c) for c in FEATURES], sp.sympify(text, locals={c: sp.Symbol(c) for c in FEATURES}), "numpy")
    return lambda Dd: np.broadcast_to(np.asarray(f(*[Dd[c].to_numpy(float) for c in FEATURES]), float), (len(Dd),))
LONG = {n: eq_fn(final_eq.loc[n, "equation"]) for n in DATASETS}

# compact equations (fitted constants); same forms as short_equations_cv.csv
SHORT_FORMS = {
    "youngs_modulus":       (lambda X, a, b, c: np.abs(a * X["S_mix"] * X["gmean_p"] + b) ** c, [2.44, 214.5, 0.709], "E = (a*S_mix*gmean_E + b)^c"),
    "density":              (lambda X, a, b: X["mean_p"] + a / X["hmean_p"] + b, [3.86, -1.61], "rho = mean_rho + a/hmean_rho + b"),
    "liquidus_temperature": (lambda X, a, b: X["mean_p"] + np.sqrt(X["S_mix"]) * (a + b / X["gmean_p"]), [-1216, 1.18e6], "T = mean_Tm + sqrt(S)*(a + b/gmean_Tm)"),
}
def fit_short(name, idx):
    f, p0, _ = SHORT_FORMS[name]; X = D[name].iloc[idx]; y = raw[name][DATASETS[name][1]].to_numpy()[idx]
    p, _ = curve_fit(lambda _, *pp: f(X, *pp).to_numpy(), np.zeros(len(y)), y, p0=p0, maxfev=20000)
    return p
SHORT_PARAMS = {}
for name in DATASETS:
    o = np.full(len(D[name]), np.nan)
    for tr, te in folds[name]:                              # constants refitted inside each training fold
        p = fit_short(name, tr); o[te] = SHORT_FORMS[name][0](D[name].iloc[te], *p)
    oof[name]["sr_short (ours)"] = o
    SHORT_PARAMS[name] = fit_short(name, np.arange(len(D[name])))
    print(f"{name:22s} {SHORT_FORMS[name][2]:40s} constants (all data): {np.round(SHORT_PARAMS[name], 5)}")

def predict(name, model, comp_df):
    Dd = descriptors(comp_df, ox[DATASETS[name][2]])
    return LONG[name](Dd) if model == "long" else SHORT_FORMS[name][0](Dd, *SHORT_PARAMS[name]).to_numpy()

## The paper's models, retrained out-of-fold on the same folds

These are the paper's best models with the hyperparameters from its Table 3:
- Young's modulus and density: random forest;
- liquidus temperature: XGBoost.

They are trained on the 17 raw oxide fractions, and every prediction is made by a model that never saw that glass. Each fold is saved as soon as it finishes.

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
PAPER_MODELS = {
    "youngs_modulus": lambda: RandomForestRegressor(n_estimators=171, max_depth=41, min_samples_split=3, min_samples_leaf=1, random_state=SEED, n_jobs=N_JOBS),
    "density":        lambda: RandomForestRegressor(n_estimators=331, max_depth=35, min_samples_split=2, min_samples_leaf=1, random_state=SEED, n_jobs=N_JOBS),
}
try:
    import xgboost as xgb
    PAPER_MODELS["liquidus_temperature"] = lambda: xgb.XGBRegressor(n_estimators=453, max_depth=15, learning_rate=0.134, min_child_weight=1,
                                                                    gamma=0.249, reg_alpha=0.838, reg_lambda=0.996, random_state=SEED, n_jobs=N_JOBS)
except ImportError:
    print("xgboost not installed: the paper's liquidus model is skipped")

for name in (DATASETS if RUN_PAPER_MODELS else []):
    if name not in PAPER_MODELS: continue
    X = raw[name][OXIDES].to_numpy(float); y = raw[name][DATASETS[name][1]].to_numpy(float)
    o = np.full(len(y), np.nan)
    for k, (tr, te) in enumerate(folds[name]):
        ck = OUT_DIR / "checkpoints" / f"paper_{name}_fold{k}.json"
        if ck.exists():
            o[te] = json.loads(ck.read_text()); print(f"{name} fold {k}: loaded from disk"); continue
        t0 = time.time(); m = PAPER_MODELS[name]().fit(X[tr], y[tr]); pred = m.predict(X[te]).astype(float)
        save_json(ck, pred.tolist()); o[te] = pred; del m; gc.collect()
        print(f"{name} fold {k}: R²={r2_score(y[te], pred):.3f}  ({(time.time() - t0) / 60:.1f} min, saved)")
    oof[name]["paper model (raw oxides)"] = o

rows = []
for name in DATASETS:
    y = raw[name][DATASETS[name][1]].to_numpy()
    for model, p in oof[name].items():
        rows.append({"property": name, "model": model, **metrics(y, p)})
single = pd.DataFrame(rows); single.to_csv(OUT_DIR / "single_property_oof_scores.csv", index=False)
single.round(4)

## Part A: the paper's experimental glass (59SiO₂–16.5Al₂O₃–10CaO–14.5MgO)

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
glass = pd.DataFrame([{o: 0.0 for o in OXIDES}]); glass.loc[0, ["SiO2", "Al2O3", "CaO", "MgO"]] = [59, 16.5, 10, 14.5]
measured = {"E (GPa)": 93.6, "rho (g/cm3)": 2.641, "E/rho (GPa cm3/g)": 93.6 / 2.641, "Tliq (K)": np.nan}
paper = {"E (GPa)": 93.311, "rho (g/cm3)": 2.603, "E/rho (GPa cm3/g)": 93.311 / 2.603, "Tliq (K)": 1621.317}
rowsA = {"measured (paper)": measured, "paper model prediction": paper}
for model in ["long", "short"]:
    E, r, T = (predict(n, model, glass)[0] for n in ["youngs_modulus", "density", "liquidus_temperature"])
    rowsA[f"our {model} equations"] = {"E (GPa)": E, "rho (g/cm3)": r, "E/rho (GPa cm3/g)": E / r, "Tliq (K)": T}
A = pd.DataFrame(rowsA).T
err = (A.drop(index="measured (paper)")[["E (GPa)", "rho (g/cm3)", "E/rho (GPa cm3/g)"]] / A.loc["measured (paper)", ["E (GPa)", "rho (g/cm3)", "E/rho (GPa cm3/g)"]] - 1) * 100
A.to_csv(OUT_DIR / "partA_experimental_glass.csv")
display(A.round(3)); print("Error vs measured (%):"); display(err.round(2))

## Part B: specific modulus E/ρ of 7,362 glasses with both properties measured

Every prediction here is out-of-fold: each glass's E and ρ were predicted by models that never saw that glass. That makes it a fair test of how well each approach predicts the combined quantity.

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
def key(df): return df[OXIDES].round(3).astype(str).agg("|".join, axis=1)
kE, kR = key(raw["youngs_modulus"]), key(raw["density"])
iE = pd.Series(np.arange(len(kE)), index=kE)[~kE.duplicated().to_numpy()]
iR = pd.Series(np.arange(len(kR)), index=kR)[~kR.duplicated().to_numpy()]
common = iE.index.intersection(iR.index)
e_idx, r_idx = iE[common].to_numpy(), iR[common].to_numpy()
E_meas = raw["youngs_modulus"]["YoungModulus"].to_numpy()[e_idx]; R_meas = raw["density"]["Density293K"].to_numpy()[r_idx]
S_meas = E_meas / R_meas
print(f"{len(common)} glasses with measured E and rho; measured E/rho: {S_meas.min():.1f}-{S_meas.max():.1f} (median {np.median(S_meas):.1f})")

rowsB, preds = [], {}
for model in oof["youngs_modulus"]:
    if model not in oof["density"]: continue
    S_pred = oof["youngs_modulus"][model][e_idx] / oof["density"][model][r_idx]; preds[model] = S_pred
    rowsB.append({"model": model, **metrics(S_meas, S_pred)})
B = pd.DataFrame(rowsB).set_index("model"); B.to_csv(OUT_DIR / "partB_specific_modulus_scores.csv")
display(B.round(4))

# which glasses would each model pick? overlap of the predicted top-10 % with the truly best 10 %
q = np.quantile(S_meas, 0.9); true_top = S_meas >= q
hit = {m: float(((p >= np.quantile(p, 0.9)) & true_top).sum() / true_top.sum()) for m, p in preds.items()}
print("Share of the truly best 10 % (by measured E/rho) found in each model's predicted top 10 %:", {k: round(v, 3) for k, v in hit.items()})

fig, axes = plt.subplots(1, len(preds), figsize=(4 * len(preds), 4)); axes = np.atleast_1d(axes)
lo, hi = np.percentile(S_meas, [0.5, 99.5])
for ax, (m, p) in zip(axes, preds.items()):
    ax.scatter(S_meas, p, s=3, alpha=0.3); ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_title(f"{m}\nR²={B.loc[m, 'R2']:.3f}", fontsize=9)
    ax.set_xlabel("measured E/ρ (GPa·cm³/g)"); ax.set_ylabel("predicted (out-of-fold)")
fig.tight_layout(); fig.savefig(OUT_DIR / "partB_specific_modulus_parity.png"); plt.show(); plt.close(fig)

## Part C: the CMSA design space (140,871 compositions), our equations vs the paper's predictions

There's no measurement here. This part checks whether our equations **agree with the paper's models** and **point to the same promising region**: ρ < 2.65 g/cm³ and E > 100 GPa (the paper's Fig. 10).

In [ ]:
assert "CONFIG_LOADED" in globals(), "Run the notebook from the top: Restart Kernel -> Run All"
P = {"E": cmsa["YoungModulus"].to_numpy(), "rho": cmsa["Density293K"].to_numpy(), "Tliq": cmsa["Tliquidus"].to_numpy()}
P["E/rho"] = P["E"] / P["rho"]
rowsC, ours_all = [], {}
for model in ["long", "short"]:
    O = {"E": predict("youngs_modulus", model, cmsa), "rho": predict("density", model, cmsa), "Tliq": predict("liquidus_temperature", model, cmsa)}
    O["E/rho"] = O["E"] / O["rho"]; ours_all[model] = O
    for q in ["E", "rho", "E/rho", "Tliq"]:
        rowsC.append({"our model": model, "quantity": q, "Pearson r": np.corrcoef(P[q], O[q])[0, 1],
                      "Spearman": spearmanr(P[q], O[q]).correlation, "mean |difference|": np.mean(np.abs(P[q] - O[q])),
                      "paper range": f"{P[q].min():.3g}-{P[q].max():.3g}", "our range": f"{O[q].min():.3g}-{O[q].max():.3g}"})
    in_paper = (P["rho"] < TARGET_REGION["max_density"]) & (P["E"] > TARGET_REGION["min_E"])
    in_ours = (O["rho"] < TARGET_REGION["max_density"]) & (O["E"] > TARGET_REGION["min_E"])
    jac = (in_paper & in_ours).sum() / max(1, (in_paper | in_ours).sum())
    topP = set(np.argsort(-P["E/rho"])[:1000]); topO = set(np.argsort(-O["E/rho"])[:1000])
    print(f"[{model}] target window: paper {in_paper.sum()} compositions, ours {in_ours.sum()}, overlap (Jaccard) {jac:.3f}; "
          f"top-1000 E/rho overlap {len(topP & topO) / 1000:.3f}")
C = pd.DataFrame(rowsC); C.to_csv(OUT_DIR / "partC_cmsa_agreement.csv", index=False); display(C.round(3))

best = int(np.argmax(ours_all["short"]["E/rho"]))
print("Highest predicted E/rho in CMSA by our short equations:",
      cmsa.loc[best, ["SiO2", "Al2O3", "CaO", "MgO"]].to_dict(), {k: round(float(v[best]), 3) for k, v in ours_all["short"].items()})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (lab, src) in zip(axes, [("paper predictions", P), ("our short equations", ours_all["short"])]):
    sc = ax.scatter(src["E"], src["rho"], c=src["Tliq"], s=1, cmap="viridis", rasterized=True)
    ax.axhline(TARGET_REGION["max_density"], color="r", lw=1); ax.axvline(TARGET_REGION["min_E"], color="r", lw=1)
    ax.set_xlabel("E (GPa)"); ax.set_ylabel("ρ (g/cm³)"); ax.set_title(lab); plt.colorbar(sc, ax=ax, label="T_liq (K)")
fig.tight_layout(); fig.savefig(OUT_DIR / "partC_cmsa_design_space.png"); plt.show(); plt.close(fig)
print("All tables and figures saved in", OUT_DIR.resolve())